# Production Dataset Validation

## Objective

The previous dataset investigation revealed significant inconsistencies in the relationship between ticket text and ticket priority.

To build a production-quality Ticket Priority Prediction system, a new customer support ticket dataset has been adopted.

This notebook validates the quality of the new dataset before any preprocessing or model development begins.

The validation includes:

- Dataset overview
- Data types
- Missing values
- Duplicate records
- Target distribution
- Language distribution
- Queue distribution
- Ticket type distribution
- Initial data quality assessment#%% md


In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)

In [2]:
DATA_PATH = Path("../data/raw/tickets_dataset.csv")

df = pd.read_csv(DATA_PATH)

print(df.shape)

df.head()

(28587, 16)


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN


## Dataset Shape

The first step is to understand the overall size of the dataset.

This includes:

- Number of rows
- Number of columns

This provides an initial indication of the dataset's scale for machine learning.

In [3]:
print("Dataset Shape:", df.shape)

Dataset Shape: (28587, 16)


## Dataset Information

Inspect the structure of the dataset.

This includes:

- Column names
- Data types
- Missing values
- Memory usage

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28587 entries, 0 to 28586
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   subject   24749 non-null  object
 1   body      28587 non-null  object
 2   answer    28580 non-null  object
 3   type      28587 non-null  object
 4   queue     28587 non-null  object
 5   priority  28587 non-null  object
 6   language  28587 non-null  object
 7   version   28587 non-null  int64 
 8   tag_1     28587 non-null  object
 9   tag_2     28574 non-null  object
 10  tag_3     28451 non-null  object
 11  tag_4     25529 non-null  object
 12  tag_5     14545 non-null  object
 13  tag_6     5874 non-null   object
 14  tag_7     2040 non-null   object
 15  tag_8     565 non-null    object
dtypes: int64(1), object(15)
memory usage: 3.5+ MB


## Column Names

List all available features in the dataset.

This helps identify:

- Text features
- Target variables
- Metadata
- Additional information useful for feature engineering

In [5]:
df.columns.tolist()

['subject',
 'body',
 'answer',
 'type',
 'queue',
 'priority',
 'language',
 'version',
 'tag_1',
 'tag_2',
 'tag_3',
 'tag_4',
 'tag_5',
 'tag_6',
 'tag_7',
 'tag_8']

## Preview Random Samples

Inspect random ticket examples to verify that the dataset contains realistic customer support conversations.

Random sampling is preferred over only viewing the first few rows because it reduces bias.

In [6]:
df.sample(5, random_state=42)

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
9284,NaN,"Kundenservice, ich möchte einen Vorfall melden...","<name>, danke für die Meldung des Vorfalls, de...",Incident,Technical Support,high,de,52,Performance,Outage,Disruption,Recovery,IT,Tech Support,NaN,NaN
2647,NaN,"Sehr geehrter Kundenservice, ich möchte detail...","Sehr geehrter <name>, vielen Dank, dass Sie un...",Request,Technical Support,high,de,52,Product,Documentation,Support,Integration,Configuration,NaN,NaN,NaN
3820,Issues with Brand Expansion,Despite enhancing ad text and boosting the bud...,I appreciate your concerns about low engagemen...,Incident,IT Support,high,en,52,Marketing,Campaign,Budget,Engagement,Brand Expansion,NaN,NaN,NaN
18831,Unusual Decline in Engagement Metrics for Camp...,noted a sudden decline in engagement metrics a...,aim to assist in addressing the sudden decline...,Incident,Returns and Exchanges,high,en,400,Performance,Feedback,Feature,IT,Tech Support,NaN,NaN,NaN
7667,Medizinische Daten in Gesundheits-IT-Lösungen,"Sehr geehrter Kundenservice, ich möchte mich n...","Sehr geehrter <name>, wir schätzen Ihr Interes...",Request,Billing and Payments,low,de,52,Security,IT,Tech Support,NaN,NaN,NaN,NaN,NaN


## Missing Values Analysis

Identify columns with missing values.

This helps determine whether cleaning or imputation will be required before model training.

In [7]:
missing = df.isnull().sum().sort_values(ascending=False)

missing[missing > 0]

tag_8      28022
tag_7      26547
tag_6      22713
tag_5      14042
subject     3838
tag_4       3058
tag_3        136
tag_2         13
answer         7
dtype: int64

## Duplicate Records

Check whether duplicate rows exist.

Duplicate records can bias the machine learning model and should be identified early.

In [8]:
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 0


## Target Distribution

Analyze the distribution of ticket priority labels.

Balanced target classes generally lead to more reliable supervised learning models.

In [9]:
df["priority"].value_counts()

priority
medium    11515
high      11178
low        5894
Name: count, dtype: int64

## Language Distribution

The dataset contains tickets in multiple languages.

Understanding the language distribution helps determine whether:

- The dataset should be filtered to a single language.
- A multilingual NLP pipeline is required.
- Language imbalance may affect model performance.

In [10]:
df["language"].value_counts()

language
en    16338
de    12249
Name: count, dtype: int64

## Queue Distribution

Analyze the distribution of support queues.

This provides an overview of the business domains represented in the dataset.

In [11]:
df["queue"].value_counts()

queue
Technical Support                  8362
Product Support                    5252
Customer Service                   4268
IT Support                         3433
Billing and Payments               2788
Returns and Exchanges              1437
Service Outages and Maintenance    1148
Sales and Pre-Sales                 918
Human Resources                     576
General Inquiry                     405
Name: count, dtype: int64

## Ticket Type Distribution

Analyze the different ticket types present in the dataset.

In [12]:
df["type"].value_counts()

type
Incident    11466
Request      8187
Problem      6012
Change       2922
Name: count, dtype: int64

## Dataset Version

Verify whether the dataset contains multiple versions.

If multiple versions exist, we may decide to keep only the latest version for training.

In [13]:
df["version"].value_counts().sort_index()

version
51       869
52      9119
400    18599
Name: count, dtype: int64

## Priority Distribution by Language

Check whether ticket priorities are reasonably distributed across languages.

In [14]:
pd.crosstab(df["language"], df["priority"])

priority,high,low,medium
language,,,
de,4832,2520,4897
en,6346,3374,6618


## Missing Subject Analysis

Determine whether missing subjects occur predominantly in a particular language or ticket type.

In [15]:
df[df["subject"].isna()][["language", "type", "queue"]].head(10)

,language,type,queue
870,en,Request,Technical Support
887,en,Incident,Technical Support
888,de,Incident,Customer Service
915,en,Change,Customer Service
929,en,Incident,Billing and Payments
933,en,Request,Service Outages and Maintenance
934,de,Incident,Product Support
946,en,Request,Customer Service
949,en,Incident,Returns and Exchanges
950,en,Request,IT Support


## Label Consistency Check

Verify whether identical ticket bodies are assigned different priority labels.

A supervised learning model assumes that the same input should not map to multiple target labels. This validation ensures that the dataset satisfies that assumption.

In [16]:
duplicate_priorities = (
    df.groupby("body")["priority"]
      .nunique()
      .sort_values(ascending=False)
)

duplicate_priorities[duplicate_priorities > 1]

Series([], Name: priority, dtype: int64)

In [17]:
print(
    "Bodies with multiple priorities:",
    (duplicate_priorities > 1).sum()
)

Bodies with multiple priorities: 0


# Validation Summary

## Dataset Assessment

The production dataset successfully passed the initial validation checks.

### Summary

- Dataset contains 28,587 support tickets.
- No duplicate records were found.
- Priority labels are well distributed across three classes.
- The dataset contains both English and German tickets.
- Ticket bodies have consistent priority labels.
- No contradictory mappings between identical ticket text and target labels were detected.

### Conclusion

Unlike the previously investigated dataset, this dataset satisfies the fundamental assumptions required for supervised machine learning.

The dataset is approved for preprocessing and model development.